# Part 2 of the project!

##### The FEMA disaster declartions will supplement the NOAA tornado CSV file. It is the "supplemental" table in the ERD. As with the tornado notebook file, we need to have a basic understanding of the data and what it represents. I have created another reference table to help navigate the CSV.

##### **FEMA Declarations Column Reference**

**Declaration Info:** `femaDeclarationString` (full declaration identifier) · `disasterNumber` (unique disaster ID) · `declarationType` (type of declaration) · `declarationTitle` (official title) · `declarationDate` (date declared) · `declarationRequestNumber` (request tracking number)

**Incident:** `incidentType` (type of disaster) · `incidentId` (unique incident ID) · `incidentBeginDate` (disaster start date) · `incidentEndDate` (disaster end date) · `designatedIncidentTypes` (all designated incident types)

**Assistance Programs:** `iaProgramDeclared` (Individual Assistance) · `ihProgramDeclared` (Individual Household) · `paProgramDeclared` (Public Assistance) · `hmProgramDeclared` (Hazard Mitigation)

**Location:** `state` · `fipsStateCode` (state FIPS code) · `fipsCountyCode` (county FIPS code) · `placeCode` (place identifier) · `designatedArea` (designated disaster area) · `region` (FEMA region)

**Administrative:** `fyDeclared` (fiscal year declared) · `tribalRequest` (tribal nation request flag) · `lastIAFilingDate` (last date to file for IA) · `disasterCloseoutDate` (disaster close date) · `lastRefresh` (last data refresh) · `hash` (data hash) · `id` (record ID)

##### **Entity Relationship Diagram**
![Tornado ERD](../Assets/Tornado_ERD.jpeg)

Now, we can import the libraries needed.

In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

##### The FEMA CSV file is much smaller, and therefore, we do not need to concat CSVs together. Instead, we can use PANDAS to read the CSV directly.

In [118]:
df = pd.read_csv('../Data/Raw_FEMA_Data/FEMA_Declarations.csv')

In [119]:
df.head()

,femaDeclarationString,disasterNumber,state,declarationType,declarationDate,fyDeclared,incidentType,declarationTitle,ihProgramDeclared,iaProgramDeclared,...,placeCode,designatedArea,declarationRequestNumber,lastIAFilingDate,incidentId,region,designatedIncidentTypes,lastRefresh,hash,id
0,FM-5529-OR,5529,OR,FM,2024-08-09T00:00:00.000Z,2024,Fire,LEE FALLS FIRE,0,0,...,99067,Washington (County),24122,NaN,2024081001,10,R,2024-08-27T18:22:14.800Z,ae87cf3c6ed795015b714af7166c7c295b2b67c7,09e3f81a-5e16-4b72-b317-1c64e0cfa59c
1,FM-5528-OR,5528,OR,FM,2024-08-06T00:00:00.000Z,2024,Fire,ELK LANE FIRE,0,0,...,99031,Jefferson (County),24116,NaN,2024080701,10,R,2024-08-27T18:22:14.800Z,432cf0995c47e3895cea696ede5621b810460501,59983f89-30bf-4888-b21b-62e8d57d9aac
2,FM-5527-OR,5527,OR,FM,2024-08-02T00:00:00.000Z,2024,Fire,MILE MARKER 132 FIRE,0,0,...,99017,Deschutes (County),24111,NaN,2024080301,10,R,2024-08-27T18:22:14.800Z,2f21d90cb6bc64b0d4121aa3f18d852bbb4b11fa,8d13ecf0-bc2f-496b-8c9f-b2e73da832a0
3,DR-4312-CA,4312,CA,DR,2017-05-02T00:00:00.000Z,2017,Severe Storm,FLOODING,0,0,...,60347,Resighini Rancheria (Indian Reservation),17035,NaN,2017041001,9,NaN,2025-03-26T20:21:32.579Z,432a3a64bdbb291ae26cf5a27a33deeabb380481,98a7c5bb-2346-45aa-a1ca-0399440d4f0b
4,DR-4251-AL,4251,AL,DR,2016-01-21T00:00:00.000Z,2016,Severe Storm,"SEVERE STORMS, TORNADOES, STRAIGHT-LINE WINDS,...",0,0,...,99001,Autauga (County),16003,NaN,2015122301,4,NaN,2025-03-27T12:21:46.559Z,dcd4ce6b37ee49875b3f1e32e9a8a16cd6a803d3,5229bbae-eee6-42b8-b277-edbafa8d6cb2


##### As seen above, there are a multitude of columns we need to delete. Based upon on the ERD, the following columns are all that is needed: disaster_number, state, year, declaration_type, ia_declared, ih_declared, incidentType.

In [120]:
cols_to_drop = ['femaDeclarationString', 'disasterNumber', 'fyDeclared',  # Define columns to drop
                    'declarationTitle', 'placeCode', 'designatedArea', 'declarationRequestNumber', 
                    'lastIAFilingDate', 'incidentId', 'region', 'designatedIncidentTypes', 'lastRefresh',
                    'hash', 'id','paProgramDeclared', 'hmProgramDeclared',
                    'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'declarationDate', 'incidentEndDate'
                    ]

FEMA_df = df.drop(columns=cols_to_drop, errors='ignore')

FEMA_df['incidentBeginDate'] = pd.to_datetime(FEMA_df['incidentBeginDate'])
FEMA_df['year'] = FEMA_df['incidentBeginDate'].dt.year
FEMA_df = FEMA_df.drop(columns=['incidentBeginDate'])
FEMA_df.head()

,state,declarationType,incidentType,ihProgramDeclared,iaProgramDeclared,year
0,OR,FM,Fire,0,0,2024
1,OR,FM,Fire,0,0,2024
2,OR,FM,Fire,0,0,2024
3,CA,DR,Severe Storm,0,0,2017
4,AL,DR,Severe Storm,0,0,2015


# <h1 style="color: orange;">Problem 1: Extra natural disasters.</h1>

##### FEMA deploys assistance to a wide variety of situations, not just tornadoes. "Fire" is not relevant for our purposes. Below we will list the incidents and decipher what needs to be kept.

In [121]:
print(FEMA_df['incidentType'].unique())

['Fire' 'Severe Storm' 'Straight-Line Winds' 'Flood' 'Hurricane'
 'Biological' 'Winter Storm' 'Tornado' 'Tropical Storm' 'Earthquake'
 'Typhoon' 'Snowstorm' 'Freezing' 'Mud/Landslide' 'Coastal Storm' 'Other'
 'Severe Ice Storm' 'Dam/Levee Break' 'Volcanic Eruption'
 'Tropical Depression' 'Toxic Substances' 'Chemical' 'Terrorist' 'Drought'
 'Human Cause' 'Fishing Losses' 'Tsunami']


In [122]:
FEMA_df = FEMA_df[FEMA_df['incidentType'].isin([ # We elected to keep the following events
    'Tornado',
    'Severe Storm',
    'Straight-Line Winds',
    'Coastal Storm'
])]

FEMA_df = FEMA_df[FEMA_df['year'].isin([2004, 2014, 2024])] # We only need to keep years 2004, 2014 and 2024

FEMA_df.head()

,state,declarationType,incidentType,ihProgramDeclared,iaProgramDeclared,year
595,MT,DR,Straight-Line Winds,0,0,2024
596,MT,DR,Straight-Line Winds,0,0,2024
608,OK,DR,Severe Storm,0,0,2024
718,NE,DR,Severe Storm,0,0,2024
719,NE,DR,Severe Storm,0,0,2024


##### For the sake of consistency, we will want to rename these columns to match the created columns in the SQL database. If we fail to do so, the will not import correctly into SQL.

In [123]:
FEMA_df = FEMA_df.rename(columns={
    'disasterNumber':   'disaster_number',
    'declarationType':  'declaration_type',
    'incidentType':     'incident_type',
    'ihProgramDeclared': 'ih_declared',
    'iaProgramDeclared': 'ia_declared'
})

FEMA_df.head()

,state,declaration_type,incident_type,ih_declared,ia_declared,year
595,MT,DR,Straight-Line Winds,0,0,2024
596,MT,DR,Straight-Line Winds,0,0,2024
608,OK,DR,Severe Storm,0,0,2024
718,NE,DR,Severe Storm,0,0,2024
719,NE,DR,Severe Storm,0,0,2024


In [124]:
FEMA_df.to_csv("../Data/Cleaned_FEMA_Data/FEMA_data_cleaned.csv", index=False)